In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE

# Identify feature groups
categorical_cols = ['source', 'browser', 'sex', 'country']
numerical_cols = ['purchase_value', 'age', 'hour_of_day', 'day_of_week', 'time_since_signup', 'device_velocity', 'ip_velocity']

X = features_df.drop(columns=['class'])
y = features_df['class']

# Fill 'Unknown' for country column if NaN remains from unmatched ranges
X['country'] = X['country'].fillna('Unknown')

# Split into train and test FIRST (To prevent data leakage during scaling/resampling)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply Transformations
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Document Distribution before Resampling
print(f"Before SMOTE - Training Set Class Distribution: {np.bincount(y_train)}")

# Apply SMOTE strictly to Training Set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_transformed, y_train)

# Document Distribution after Resampling
print(f"After SMOTE - Training Set Class Distribution: {np.bincount(y_train_resampled)}")

# Save outputs to data/processed/
np.save('../data/processed/X_train_resampled.npy', X_train_resampled)
np.save('../data/processed/y_train_resampled.npy', y_train_resampled)
np.save('../data/processed/X_test.npy', X_test_transformed)
np.save('../data/processed/y_test.npy', y_test)